# Chapter 10, Exercise 4: A cascaded SLU pipeline (Whisper then a zero-shot LLM)

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 10, Exercise 4.** Design a cascaded spoken-language-understanding pipeline: transcribe 10 dialectal audio files, then write a zero-shot prompt instructing a large language model to act as a flight-booking assistant and return a fixed JSON schema, for example {"intent": "book_flight", "slots": {"destination": "Riyadh", "date": "tomorrow"}}. Report the invalid-JSON rate, the missing-slot rate, and the wrong-slot-value rate against the ground truth, discuss whether failures come from recognition errors or prompt design, and compare this large-language-model cascade with an end-to-end approach when labeled dialect data is limited.

**Note on data and models.** Ten human-recorded Arabic travel requests are taken from Speech-MASSIVE (`ar-SA`, CC BY-NC-SA 4.0; the Arabic text is localized with Gulf features such as حقي, بكرة, ايش) with their gold intent and slots mapped onto a travel-assistant schema. The LLM is an open instruction-tuned model run locally (`Qwen/Qwen2.5-1.5B-Instruct` by default; any chat model on the Hub, or an API model if you add a key, can be substituted). Results depend on the recognizer and the LLM used and are reported as measured.

## Requirements

Runs on CPU with `openai/whisper-small` and `Qwen/Qwen2.5-1.5B-Instruct` (about 20 LLM calls; expect 10 to 20 minutes on the free CPU tier). A T4 GPU makes it a couple of minutes and allows a larger LLM.

## Testing status

Executed end to end on CPU with the default models; the reported rates are what that run produced and will change with other models.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/solutions/Chapter_10_Exercise_04.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

In [1]:
!pip install -q "transformers>=4.40" torch accelerate jiwer pandas pyarrow huggingface_hub soundfile librosa

## 1. Ten dialectal travel requests with ground truth

MASSIVE's transport intents are mapped to the assistant's schema: `transport_ticket` to `book_ticket`, `transport_taxi` to `book_taxi`, `transport_query` and `transport_traffic` to `transport_info`; anything else is `other`. Slots kept: `destination` (from `place_name`), `date`, `time`, `transport_type`. Two non-transport utterances are included as out-of-domain controls (gold intent `other`, no slots).

In [2]:
import io, os, re, json, numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf, librosa
from huggingface_hub import hf_hub_download

p = hf_hub_download("FBK-MT/Speech-MASSIVE", "ar-SA/train_115-00000-of-00001.parquet", repo_type="dataset")
tab = pq.ParquetFile(p).read().to_pandas()
INTENT_MAP = {"transport_ticket": "book_ticket", "transport_taxi": "book_taxi", "transport_query": "transport_info", "transport_traffic": "transport_info"}
SLOT_MAP = {"place_name": "destination", "date": "date", "time": "time", "transport_type": "transport_type"}
def gold_slots(annot):
    d = {}
    for m in re.finditer(r"\[([^:\]]+) : ([^\]]+)\]", annot):
        s, v = m.group(1).strip(), m.group(2).strip()
        if s in SLOT_MAP and SLOT_MAP[s] not in d: d[SLOT_MAP[s]] = v
    return d
transport = tab[tab["scenario_str"] == "transport"].head(8)
others = tab[tab["scenario_str"] != "transport"].sample(2, random_state=0)
data = pd.concat([transport, others]).reset_index(drop=True)
data["gold_intent"] = data["intent_str"].map(lambda s: INTENT_MAP.get(s, "other"))
data["gold_slots"] = data.apply(lambda r: gold_slots(r["annot_utt"]) if r["gold_intent"] != "other" else {}, axis=1)
os.makedirs("slu10", exist_ok=True)
for i, r in data.iterrows():
    y, sr = sf.read(io.BytesIO(r["audio"]["bytes"])); sf.write(f"slu10/utt_{i:02d}.wav", y, sr)
data[["utt", "gold_intent", "gold_slots", "speaker_sex"]]

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,utt,gold_intent,gold_slots,speaker_sex
0,أقرب محطة قطار,transport_info,{'destination': 'محطة قطار'},Female
1,سيارة المطار,book_taxi,"{'transport_type': 'سيارة', 'destination': 'ال...",Female
2,الاتصال سيارة,book_taxi,{'transport_type': 'سيارة'},Male
3,قطار الحرمين يوصل جدة,transport_info,{'destination': 'جدة'},Male
4,كيف حال المواصلات خارج المكتب حقي,transport_info,{'destination': 'المكتب'},Male
5,قطار,book_ticket,{'transport_type': 'قطار'},Female
6,أوبر الساعة العاشرة مساء الليلة,book_taxi,{'time': 'الساعة العاشرة مساء'},Male
7,الرجعة رحلة الخبر إلى الرياض قطار,book_ticket,"{'destination': 'الخبر', 'transport_type': 'قط...",Female
8,ألعاب تعليمية,other,{},Female
9,أعطنا أخبار انتخاب الرئيس,other,{},Male


## 2. Stage 1: transcribe (Whisper)

In [3]:
import torch, warnings, transformers, jiwer, unicodedata
warnings.filterwarnings("ignore"); transformers.logging.set_verbosity_error()
from transformers import pipeline
ASR = os.environ.get("ASR_MODEL", "openai/whisper-small")
asr = pipeline("automatic-speech-recognition", model=ASR, device=0 if torch.cuda.is_available() else -1,
               generate_kwargs={"language": "arabic", "task": "transcribe"})
hyps = []
for i, r in data.iterrows():
    y, sr = sf.read(f"slu10/utt_{i:02d}.wav")
    if y.ndim > 1: y = y.mean(axis=1)
    if sr != 16000: y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=16000)
    hyps.append(asr({"raw": y.astype(np.float32), "sampling_rate": 16000})["text"].strip())
data["asr"] = hyps
DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
def norm(t):
    t = unicodedata.normalize("NFC", t).replace("\u0640", ""); t = DIAC.sub("", t)
    t = re.sub(r"[\u060C\u061B\u061F!-/:-@\[-`{-~]", " ", t); t = re.sub("[أإآٱ]", "ا", t).replace("ة", "ه").replace("ى", "ي")
    return re.sub(r"\s+", " ", t).strip()
o = jiwer.process_words([norm(t) for t in data["utt"]], [norm(h) for h in data["asr"]])
print(f"ASR = {ASR}; WER on the 10 utterances = {100*o.wer:.1f}% (S={o.substitutions} D={o.deletions} I={o.insertions})")
data[["utt", "asr"]]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:  24%|██▎       | 113/479 [00:00<00:00, 1088.54it/s]

Loading weights:  48%|████▊     | 231/479 [00:00<00:00, 1139.30it/s]

Loading weights:  72%|███████▏  | 346/479 [00:00<00:00, 1105.53it/s]

Loading weights:  95%|█████████▌| 457/479 [00:00<00:00, 1101.35it/s]

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1119.05it/s]

ASR = openai/whisper-small; WER on the 10 utterances = 40.0% (S=14 D=0 I=0)


,utt,asr
0,أقرب محطة قطار,أكرم حطات كتار
1,سيارة المطار,سيارة المطار
2,الاتصال سيارة,الاتصال سيارة
3,قطار الحرمين يوصل جدة,كتار الحرمين يوصل جدا
4,كيف حال المواصلات خارج المكتب حقي,كيف حال الموصلات خارج المكتب حقي؟
5,قطار,كوستور
6,أوبر الساعة العاشرة مساء الليلة,وبر الساعة العشرة مساء الليلة
7,الرجعة رحلة الخبر إلى الرياض قطار,رجع رحلة الخبر إلى الرياض كتار
8,ألعاب تعليمية,لعب تعلمية
9,أعطنا أخبار انتخاب الرئيس,اعطينا اخبار انتخاب الرئيس


## 3. Stage 2: zero-shot LLM with a fixed JSON schema

The prompt fixes the intent inventory, the slot names, the output format (JSON only, no prose) and the rule for absent slots. The same prompt is used for every utterance. Two conditions are run: the LLM on the **ASR transcript** (the real cascade) and on the **gold transcript** (an oracle that isolates prompt and LLM failures from recognition failures).

In [4]:
SYSTEM = (
 "You are a travel-booking assistant for Arabic speakers. The user speaks Gulf or other Arabic dialects.\n"
 "Read the user request and return ONLY a JSON object, with no explanation, in exactly this schema:\n"
 '{"intent": "<one of: book_ticket, book_taxi, transport_info, other>", '
 '"slots": {"destination": "<string or null>", "date": "<string or null>", "time": "<string or null>", "transport_type": "<string or null>"}}\n'
 "Copy slot values in Arabic exactly as they appear in the request. Use null for a slot that is not mentioned. "
 "If the request is not about travel, use intent \"other\" and set every slot to null.")

LLM = os.environ.get("LLM_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(LLM)
# bfloat16 halves the memory (about 3 GB for the 1.5 B model) and runs on both GPU and modern CPUs
llm = AutoModelForCausalLM.from_pretrained(LLM, torch_dtype=torch.bfloat16,
                                           device_map="auto" if torch.cuda.is_available() else None).eval()

def ask(text, max_new_tokens=120):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    enc = {k: v.to(llm.device) for k, v in enc.items()}
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

data["llm_on_asr"] = [ask(t) for t in data["asr"]]
data["llm_on_gold"] = [ask(t) for t in data["utt"]]
data[["asr", "llm_on_asr"]]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:  12%|█▏        | 40/338 [00:00<00:00, 396.66it/s]

Loading weights:  25%|██▌       | 86/338 [00:00<00:00, 424.34it/s]

Loading weights:  38%|███▊      | 129/338 [00:00<00:00, 402.63it/s]

Loading weights:  50%|█████     | 170/338 [00:00<00:00, 378.05it/s]

...

Loading weights:  75%|███████▌  | 255/338 [00:00<00:00, 391.17it/s]

Loading weights:  87%|████████▋ | 295/338 [00:00<00:00, 377.99it/s]

Loading weights: 100%|█████████▉| 337/338 [00:00<00:00, 390.45it/s]

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 390.37it/s]

,asr,llm_on_asr
0,أكرم حطات كتار,"{\n ""intent"": ""other"",\n ""slots"": {\n ""de..."
1,سيارة المطار,"{""intent"": ""book_taxi"", ""slots"": {""destination..."
2,الاتصال سيارة,"{""intent"": ""book_taxi"", ""slots"": {""transport_t..."
3,كتار الحرمين يوصل جدا,"{""intent"": ""book_taxi"", ""slots"": {""destination..."
4,كيف حال الموصلات خارج المكتب حقي؟,"{\n ""intent"": ""book_taxi"",\n ""slots"": {\n ..."
5,كوستور,"{\n ""intent"": ""other"",\n ""slots"": {\n ""de..."
6,وبر الساعة العشرة مساء الليلة,"{""intent"": ""book_taxi"", ""slots"": {""time"": ""الع..."
7,رجع رحلة الخبر إلى الرياض كتار,"{""intent"": ""book_ticket"", ""slots"": {""destinati..."
8,لعب تعلمية,{}
9,اعطينا اخبار انتخاب الرئيس,"{\n ""intent"": ""other"",\n ""slots"": {\n ""de..."


## 4. Scoring: invalid JSON, missing slots, wrong slot values

* **Invalid-JSON rate**: outputs that cannot be parsed into an object with the required keys.
* **Missing-slot rate**: over the gold slots, the share the model returned as null or omitted.
* **Wrong-slot-value rate**: over the gold slots the model did fill, the share whose value does not match the gold value after normalization (the normalization is stated: diacritics and punctuation removed, alif/ة/ى unified, the definite article ignored).
* Intent accuracy is reported as well.

In [5]:
def parse_json(s):
    s = s.strip()
    s = re.sub(r"^```(?:json)?|```$", "", s, flags=re.M).strip()
    m = re.search(r"\{.*\}", s, flags=re.S)
    if not m: return None
    try:
        obj = json.loads(m.group(0))
    except Exception:
        return None
    if not isinstance(obj, dict) or "intent" not in obj or not isinstance(obj.get("slots"), dict): return None
    return obj

def slot_norm(v):
    if v is None: return None
    v = norm(str(v)); v = re.sub(r"^(ال)", "", v).strip()
    return v or None

def score(column):
    n_invalid = 0; gold_total = 0; missing = 0; filled = 0; wrong = 0; intent_ok = 0
    for _, r in data.iterrows():
        obj = parse_json(r[column])
        if obj is None:
            n_invalid += 1; gold_total += len(r["gold_slots"]); missing += len(r["gold_slots"]); continue
        if str(obj.get("intent")).strip() == r["gold_intent"]: intent_ok += 1
        for s, gv in r["gold_slots"].items():
            gold_total += 1
            pv = obj["slots"].get(s) if isinstance(obj["slots"], dict) else None
            if pv in (None, "", "null"): missing += 1
            else:
                filled += 1
                if slot_norm(pv) != slot_norm(gv): wrong += 1
    return {"invalid JSON %": 100*n_invalid/len(data), "intent accuracy %": 100*intent_ok/len(data),
            "missing-slot % (of gold slots)": 100*missing/gold_total if gold_total else 0,
            "wrong-slot-value % (of filled gold slots)": 100*wrong/filled if filled else 0,
            "gold slots": gold_total}

res = pd.DataFrame({"LLM on ASR transcript": score("llm_on_asr"), "LLM on gold transcript": score("llm_on_gold")}).round(1)
print("ASR:", ASR, "| LLM:", LLM); res

ASR: openai/whisper-small | LLM: Qwen/Qwen2.5-1.5B-Instruct


,LLM on ASR transcript,LLM on gold transcript
invalid JSON %,10.0,0.0
intent accuracy %,50.0,50.0
missing-slot % (of gold slots),20.0,10.0
wrong-slot-value % (of filled gold slots),50.0,33.3
gold slots,10.0,10.0


In [6]:
# Side-by-side view for the error analysis
for _, r in data.iterrows():
    print("GOLD :", r["utt"], "|", r["gold_intent"], r["gold_slots"])
    print("ASR  :", r["asr"]); print("LLM(asr) :", r["llm_on_asr"].replace("\n", " ")); print("LLM(gold):", r["llm_on_gold"].replace("\n", " ")); print()

GOLD : أقرب محطة قطار | transport_info {'destination': 'محطة قطار'}
ASR  : أكرم حطات كتار
LLM(asr) : {   "intent": "other",   "slots": {     "destination": null,     "date": null,     "time": null,     "transport_type": null   } }
LLM(gold): {"intent": "book_taxi", "slots": {"transport_type": "taxi"}}

GOLD : سيارة المطار | book_taxi {'transport_type': 'سيارة', 'destination': 'المطار'}
ASR  : سيارة المطار
LLM(asr) : {"intent": "book_taxi", "slots": {"destination": "المطار", "date": null, "time": null, "transport_type": "سيارة"}}
LLM(gold): {"intent": "book_taxi", "slots": {"destination": "المطار", "date": null, "time": null, "transport_type": "سيارة"}}

GOLD : الاتصال سيارة | book_taxi {'transport_type': 'سيارة'}
ASR  : الاتصال سيارة
LLM(asr) : {"intent": "book_taxi", "slots": {"transport_type": "سيارة"}}
LLM(gold): {"intent": "book_taxi", "slots": {"transport_type": "سيارة"}}

GOLD : قطار الحرمين يوصل جدة | transport_info {'destination': 'جدة'}
ASR  : كتار الحرمين يوصل جدا
LLM(asr) : 

## 5. Where do the failures come from?

Compare the two columns of the results table. Failures that appear **only on the ASR transcript** are recognition errors propagating through the cascade (Figure 10.2): a misheard place name becomes a wrong slot value, a dropped time expression becomes a missing slot. Failures that appear **on the gold transcript too** are prompt or model failures: invalid JSON (prose around the object, a wrong key, a code fence), an intent chosen from outside the inventory, slot values translated into English or MSA when the prompt asked for verbatim Arabic, or dialect words (بكرة 'tomorrow', الحين 'now') not recognized as dates and times. Each kind has its own fix: recognition errors call for a dialect-adapted recognizer or n-best hypotheses passed to the LLM; format errors call for a stricter prompt, a JSON-schema-constrained decoder or a repair step; value errors call for a normalization and grounding stage after extraction (Table 10.5).

## 6. LLM cascade versus end-to-end SLU when labeled dialect data is limited

| | LLM cascade (ASR then zero-shot LLM) | End-to-end SLU (speech to intent and slots) |
|---|---|---|
| Labeled dialect data needed | none for the understanding stage (zero-shot); the recognizer still benefits from dialect adaptation | hundreds to thousands of labeled dialect utterances per domain, which for most dialects do not exist (Speech-MASSIVE Arabic offers 115 training recordings, Section 10.9) |
| Error propagation | recognition errors pass into the LLM unchanged; visible and attributable because the transcript exists | no explicit transcript hand-off; errors are harder to attribute |
| Adaptation to a new domain or schema | edit the prompt; minutes | re-annotate and retrain |
| Output format guarantees | none by construction; must be enforced by prompting, constrained decoding or validation (the invalid-JSON rate above) | fixed by the model's output layer |
| Privacy and cost | transcripts may leave the device to an LLM service (Section 10.8); latency of a large model | can run on device; no text leaves |
| Ceiling | limited by ASR quality on dialect and by the LLM's dialect knowledge | can exploit acoustic cues a transcript loses, and specialized models often beat LLMs on in-domain benchmarks, but only with data |

With little labeled dialect data the cascade is the pragmatic choice: it works today, its failures are inspectable, and its main weakness (dialect recognition) is exactly where foundation-model adaptation (Chapter 6) can be applied without SLU labels. The end-to-end route becomes attractive once a few thousand labeled dialect utterances exist, at which point it should be compared on the same test set with intent accuracy, slot F1 and, for multi-turn use, joint goal accuracy, per dialect.